In [ ]:
!pip install requests

1. Tasks to complete
Task 1 — Basic website data extraction
Easy
 After Chapters 1–2


Objective: Extract information from a website using Python HTTP requests and HTML parsing.

Website: Books to Scrape (a practice website designed for scraping).

Requirements:
- Fetch the homepage using Python.
- Extract all book titles.
- Extract book prices.
- Extract availability.
- Extract star ratings.
- Print the results in a readable format.

In [ ]:
import requests 
from bs4 import BeautifulSoup
from requests import HTTPError 
from urllib.parse import urljoin

visited_links = set()
pageLinks = {'index.html'}

def scrapePage(link):
    base_url = 'https://books.toscrape.com/'
    full_url = urljoin(base_url, link)

    if full_url in visited_links:
        return
    print(f"--- Scraping: {full_url} ---")
    try:
        response = requests.get(full_url)
        bs = BeautifulSoup(response.text, 'html.parser')
        for book in bs.find_all('article', {'class': 'product_pod'}):
            title = book.find('h3').get_text()
            print('Title: ' + title)
            price = book.find('div', {'class': 'product_price'}).find('p', {'class': 'price_color'}).get_text()
            price_stripped = price[2:]
            print('Price: ' + price_stripped)
            availability = book.find('p', {'class': 'instock availability'}).get_text().strip()
            print('Availability: ' + availability)
            rating_classes = book.find('p', {'class': 'star-rating'})['class']
            rating = rating_classes[1] if len(rating_classes) > 1 else 'Unknown'
            print('Rating:', rating)
            print("-" * 40)

        # Find the next page link from the main page layout (NOT inside the book loop)
        next_button = bs.find('li', {'class': 'next'})
        if next_button and next_button.find('a'):
            next_href = next_button.find('a')['href']

            # Resolve relative path based on the current page's directory structure
            next_page_url = urljoin(full_url, next_href)

            relative_next = next_page_url.replace(base_url, "")

            if relative_next not in visited_links:
                pageLinks.add(relative_next)

    except HTTPError as h:
        return None
    except AttributeError as e:
        return None

while len(pageLinks) > 0:
    current_link = pageLinks.pop()
    scrapePage(current_link)

In [ ]:
!pip install pandas openpyxl

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd

# Create a list to store all scraped books
all_books = []

url = 'https://books.toscrape.com/catalogue/page-1.html'

while url:
    print(f"Scraping: {url}")
    response = requests.get(url)
    
    if response.status_code != 200:
        break
        
    bs = BeautifulSoup(response.text, 'html.parser')
    
    for book in bs.find_all('article', {'class': 'product_pod'}):
        title = book.find('h3').a['title']
        price = book.find('p', {'class': 'price_color'}).get_text()[2:] # removes the '£' symbol
        availability = book.find('p', {'class': 'instock availability'}).get_text().strip()
        
        rating_classes = book.find('p', {'class': 'star-rating'})['class']
        rating = rating_classes[1] if len(rating_classes) > 1 else 'Unknown'
        
        all_books.append({
            'Title': title,
            'Price': price,
            'Availability': availability,
            'Rating': rating
        })
        
    # Find the next page link dynamically
    next_button = bs.select_one('li.next > a')
    if next_button:
        url = urljoin(url, next_button['href'])
    else:
        url = None

# Once the loop finishes, convert the list to a pandas DataFrame
df = pd.DataFrame(all_books)

# Save the DataFrame to an Excel file
df.to_excel('books_data.xlsx', index=False, sheet_name='Books')

print("Scraping complete! Data successfully saved to 'books_data.xlsx'.")

Scraping: https://books.toscrape.com/catalogue/page-1.html
Scraping: https://books.toscrape.com/catalogue/page-2.html
Scraping: https://books.toscrape.com/catalogue/page-3.html
Scraping: https://books.toscrape.com/catalogue/page-4.html
Scraping: https://books.toscrape.com/catalogue/page-5.html
Scraping: https://books.toscrape.com/catalogue/page-6.html
Scraping: https://books.toscrape.com/catalogue/page-7.html
Scraping: https://books.toscrape.com/catalogue/page-8.html
Scraping: https://books.toscrape.com/catalogue/page-9.html
Scraping: https://books.toscrape.com/catalogue/page-10.html
Scraping: https://books.toscrape.com/catalogue/page-11.html
Scraping: https://books.toscrape.com/catalogue/page-12.html
Scraping: https://books.toscrape.com/catalogue/page-13.html
Scraping: https://books.toscrape.com/catalogue/page-14.html
Scraping: https://books.toscrape.com/catalogue/page-15.html
Scraping: https://books.toscrape.com/catalogue/page-16.html
Scraping: https://books.toscrape.com/catalogue/pa